# Business Entity Resolution — ML Matching & Validation (Turn 5)

This is a **thin orchestration notebook** designed for Google Colab and local Python environments.
The reusable logic lives inside `src/`.

### Pipeline Overview
1. **Environment Diagnostics**: Python version, available RAM/Disk, GPU.
2. **Data & Candidate Loading**: Inverted index blocking -> candidate pairs -> ground truth labeling.
3. **Pairwise Feature Matrix**: 21 deterministic numerical features.
4. **Entity-Level Train/Validation Split**: 80/20 GroupSplit by `source1_entity_id` to guarantee zero data leakage.
5. **Model 1: Baseline Logistic Regression** (standardized features + balanced weighting).
6. **Model 2: Gradient Boosted Trees (LightGBM)** (MIT licensed, <= 8B parameters).
7. **Validation Threshold Sweep**: Search optimal $\tau \in [0.05, 0.95]$ to maximize competition entity-level macro $F_{0.5}$ ($\beta=0.5$).
8. **Error Analysis**: Separate candidate-generation (blocking) errors from classifier errors.
9. **Artifact Persistence**: Save best model and threshold configuration for Turn 6 test inference.

In [ ]:
# CELL 1: Environment Information
import os
import platform
import shutil
import sys

print("=" * 60)
print("GOOGLE COLAB / RUNTIME ENVIRONMENT")
print("=" * 60)
print(f"Python Version:   {sys.version.split()[0]}")
print(f"Platform:         {platform.platform()}")
print(f"CPU Count:        {os.cpu_count()}")

try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"Total RAM:        {vm.total / (1024**3):.2f} GB (Available: {vm.available / (1024**3):.2f} GB)")
except ImportError:
    print("Total RAM:        psutil not available")

disk = shutil.disk_usage(".")
print(f"Available Disk:   {disk.free / (1024**3):.2f} GB")

try:
    import subprocess
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    if gpu.returncode == 0 and gpu.stdout.strip():
        print(f"GPU Device:       {gpu.stdout.strip().splitlines()[0]}")
    else:
        print("GPU Device:       None (CPU execution)")
except Exception:
    print("GPU Device:       None (CPU execution)")
print("=" * 60)

In [ ]:
# CELL 2: Install Requirements
# In Google Colab, uncomment and run:
# !pip install -q pandas numpy scikit-learn lightgbm joblib pytest psutil

import lightgbm
import numpy
import pandas
import sklearn

print("Installed Package Versions:")
print(f"  pandas:       {pandas.__version__}")
print(f"  numpy:        {numpy.__version__}")
print(f"  scikit-learn: {sklearn.__version__}")
print(f"  lightgbm:     {lightgbm.__version__}")

In [ ]:
# CELL 3: Clone / Load Project & Ensure Imports
import sys
from pathlib import Path

# In Google Colab, if cloning from GitHub:
# !git clone <repo_url>
# %cd business_entity_resolution

PROJECT_ROOT = Path().resolve()
if (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root on sys.path: {PROJECT_ROOT}")

from src.config import Config
from src.blocking import BlockingConfig, build_candidates
from src.pair_builder import build_train_pairs
from src.features import compute_features, DEFAULT_FEATURE_NAMES
from src.model import entity_train_val_split, LogisticRegressionMatcher, LightGBMMatcher
from src.threshold import find_optimal_threshold, evaluate_at_threshold
from src.prediction import predict_matches
from src.evaluation import ground_truth_to_dict, analyze_validation_errors

print("All src modules successfully imported.")

In [ ]:
# CELL 4: Configure Dataset Location & Optional Google Drive Mounting
import os
from pathlib import Path

# Optional Google Drive mount for persistent caching across Colab restarts:
# from google.colab import drive
# drive.mount('/content/drive')
# os.environ["BER_CACHE_DIR"] = "/content/drive/MyDrive/BER_Hackathon/cache"

cfg = Config()
print(f"Train Source 1:     {cfg.TRAIN_SOURCE1} (exists: {cfg.TRAIN_SOURCE1.exists()})")
print(f"Train Ground Truth: {cfg.TRAIN_GROUND_TRUTH} (exists: {cfg.TRAIN_GROUND_TRUTH.exists()})")
print(f"Cache Directory:    {cfg.CACHE_DIR}")
print(f"Output Directory:   {cfg.OUTPUT_DIR}")

In [ ]:
# CELL 5: Load / Build Cached Candidate Data
import pandas as pd
from scripts.benchmark_blocking import load_or_create_benchmark_sample

SAMPLE_S1 = 100
DISTRACTORS = 500

s1, s2, s3, gt_df = load_or_create_benchmark_sample(
    sample_s1_count=SAMPLE_S1,
    distractor_count=DISTRACTORS,
    cache_dir=cfg.CACHE_DIR,
)
gt_dict = ground_truth_to_dict(gt_df)

pairs_pkl = cfg.CACHE_DIR / f"train_pairs_s1_{SAMPLE_S1}_dist_{DISTRACTORS}.pkl"
if pairs_pkl.exists():
    print(f"Loading cached candidate pairs from {pairs_pkl}...")
    pairs = pd.read_pickle(pairs_pkl)
else:
    print("Generating candidate pairs via 4-stage blocking...")
    candidates = build_candidates(s1, s2, s3, config=cfg)
    print("Labeling pairs with ground truth (positives + hard negatives)...")
    pairs = build_train_pairs(s1, s2, s3, candidates, gt_df, random_seed=42)
    pairs.to_pickle(pairs_pkl)

pos_count = (pairs["label"] == 1).sum()
neg_count = (pairs["label"] == 0).sum()
print(f"Candidate pairs ready: {len(pairs)} rows ({pos_count} positives, {neg_count} hard negatives)")

In [ ]:
# CELL 6: Build / Load Pairwise Features
feats_pkl = cfg.CACHE_DIR / f"features_s1_{SAMPLE_S1}_dist_{DISTRACTORS}.pkl"
if feats_pkl.exists():
    print(f"Loading cached feature matrix from {feats_pkl}...")
    features_df = pd.read_pickle(feats_pkl)
else:
    print("Computing 21 pairwise numerical features...")
    features_df = compute_features(pairs)
    features_df.to_pickle(feats_pkl)

# Join features into pairs DataFrame
for col in DEFAULT_FEATURE_NAMES:
    if col in features_df.columns:
        pairs[col] = features_df[col].values

print(f"Feature matrix shape: {features_df.shape} (21 features)")
print("Features:", DEFAULT_FEATURE_NAMES[:5], "...")

In [ ]:
# CELL 7: Create Entity-Level Train / Validation Split
# Group split strictly by Source 1 entity: 80% train, 20% validation
train_pairs, val_pairs, train_gt, val_gt = entity_train_val_split(
    pairs, gt_dict, val_size=0.20, random_state=42
)

train_s1 = set(train_pairs["s1_id"].unique()) | set(train_gt.keys())
val_s1 = set(val_pairs["s1_id"].unique()) | set(val_gt.keys())

# Safety verification: zero data leakage
assert len(train_s1 & val_s1) == 0, "DATA LEAKAGE DETECTED! S1 sets overlap!"
print(f"Train Entities: {len(train_s1)} | Train Pairs: {len(train_pairs)} (Pos: {(train_pairs['label']==1).sum()}, Neg: {(train_pairs['label']==0).sum()})")
print(f"Val Entities:   {len(val_s1)} | Val Pairs:   {len(val_pairs)} (Pos: {(val_pairs['label']==1).sum()}, Neg: {(val_pairs['label']==0).sum()})")
print(f"Data Leakage Verification: intersection(train_s1, val_s1) == empty -> PASSED (0 overlapping entities)")

In [ ]:
# CELL 8: Train Logistic Regression Baseline
import time

lr_start = time.time()
lr_matcher = LogisticRegressionMatcher(random_state=42)
lr_matcher.fit(train_pairs, train_pairs["label"])
lr_train_time = round(time.time() - lr_start, 3)

print(f"Logistic Regression baseline trained in {lr_train_time}s")

In [ ]:
# CELL 9: Evaluate Logistic Regression on Validation Set
lr_val_proba = lr_matcher.predict_proba(val_pairs)
lr_best_thresh, lr_best_macro_f05, lr_history = find_optimal_threshold(
    val_pairs, lr_val_proba, val_gt
)
lr_metrics = evaluate_at_threshold(val_pairs, lr_val_proba, lr_best_thresh, val_gt)
lr_metrics["runtime_s"] = lr_train_time

print(f"LR Optimal Validation Threshold: {lr_best_thresh:.2f}")
print(f"LR Validation Macro F0.5:        {lr_best_macro_f05:.4f}")
print(f"LR Pair Precision:               {lr_metrics['pair_precision']:.4f}")
print(f"LR Pair Recall:                  {lr_metrics['pair_recall']:.4f}")
print(f"LR Singleton Accuracy:           {lr_metrics['singleton_accuracy']:.4f}")

In [ ]:
# CELL 10: Train Gradient Boosting (LightGBM)
lgb_start = time.time()
lgb_matcher = LightGBMMatcher(n_estimators=100, learning_rate=0.05, random_state=42)
lgb_matcher.fit(train_pairs, train_pairs["label"])
lgb_train_time = round(time.time() - lgb_start, 3)

print(f"LightGBM gradient booster trained in {lgb_train_time}s")

In [ ]:
# CELL 11: Evaluate Gradient Boosting on Validation Set
lgb_val_proba = lgb_matcher.predict_proba(val_pairs)
lgb_best_thresh, lgb_best_macro_f05, lgb_history = find_optimal_threshold(
    val_pairs, lgb_val_proba, val_gt
)
lgb_metrics = evaluate_at_threshold(val_pairs, lgb_val_proba, lgb_best_thresh, val_gt)
lgb_metrics["runtime_s"] = lgb_train_time

print(f"LightGBM Optimal Validation Threshold: {lgb_best_thresh:.2f}")
print(f"LightGBM Validation Macro F0.5:        {lgb_best_macro_f05:.4f}")
print(f"LightGBM Pair Precision:               {lgb_metrics['pair_precision']:.4f}")
print(f"LightGBM Pair Recall:                  {lgb_metrics['pair_recall']:.4f}")
print(f"LightGBM Singleton Accuracy:           {lgb_metrics['singleton_accuracy']:.4f}")

In [ ]:
# CELL 12: Validation Threshold Sweep Comparison
print(f"{'Threshold':<10} | {'LR Macro F0.5':<15} | {'LightGBM Macro F0.5':<20}")
print("-" * 52)
for thresh in sorted(lr_history.keys()):
    print(f"{thresh:<10.2f} | {lr_history[thresh]:<15.4f} | {lgb_history.get(thresh, 0.0):<20.4f}")

In [ ]:
# CELL 13: Entity-Level Macro F0.5 Comparison Table
print("=" * 88)
print("MODEL COMPARISON TABLE (Held-Out Entity Validation)")
print("=" * 88)
header = f"{'Model':<22} | {'Thresh':<6} | {'Pair Prec':<9} | {'Pair Rec':<8} | {'Pair F1':<7} | {'PR-AUC':<6} | {'Macro F0.5':<10} | {'Singl Acc':<9} | {'Runtime':<7}"
print(header)
print("-" * 88)
for name, m in [("Logistic Regression", lr_metrics), ("LightGBM (GBDT)", lgb_metrics)]:
    row = f"{name:<22} | {m['threshold']:<6.2f} | {m['pair_precision']:<9.4f} | {m['pair_recall']:<8.4f} | {m['pair_f1']:<7.4f} | {m['pr_auc']:<6.4f} | {m['macro_f05']:<10.4f} | {m['singleton_accuracy']:<9.4f} | {m['runtime_s']:<6.2f}s"
    print(row)
print("=" * 88)

In [ ]:
# CELL 14: Validation Error Analysis & Feature Importance
best_model = lgb_matcher if lgb_best_macro_f05 >= lr_best_macro_f05 else lr_matcher
best_thresh = lgb_best_thresh if lgb_best_macro_f05 >= lr_best_macro_f05 else lr_best_thresh
best_proba = lgb_val_proba if lgb_best_macro_f05 >= lr_best_macro_f05 else lr_val_proba

val_preds = predict_matches(val_pairs, best_proba, threshold=best_thresh, s1_entities=val_gt.keys())
error_report = analyze_validation_errors(val_gt, val_preds, val_pairs, best_proba, best_thresh)

print("Error Analysis Diagnostics:")
print(f"  Total Validation Entities: {error_report['total_val_entities']}")
print(f"  Total True Matches:        {error_report['total_true_matches']}")
print(f"  Correct True Positives:    {error_report['true_positives']}")
print(f"  False Positives:           {error_report['false_positives']}")
print(f"  Candidate Generation Miss: {error_report['blocking_misses_count']} ({error_report['blocking_miss_rate_pct']:.2f}%)")
print(f"  Classifier Miss (P < {best_thresh:.2f}):  {error_report['classifier_misses_count']} ({error_report['classifier_miss_rate_pct']:.2f}%)")
print(f"  Singleton Accuracy:        {error_report['singleton_accuracy_pct']:.2f}%")

print("\nTop 5 LightGBM Features by Gain:")
lgb_fi = lgb_matcher.get_feature_importances(importance_type="gain")
for rank, (feat, gain) in enumerate(sorted(lgb_fi.items(), key=lambda x: x[1], reverse=True)[:5], 1):
    print(f"  {rank}. {feat:<28}: {gain*100:.2f}%")

In [ ]:
# CELL 15: Save Selected Model and Threshold Configuration
import json

best_model_path = cfg.CACHE_DIR / "best_matcher.joblib"
best_model.save(best_model_path)

threshold_cfg = {
    "best_model": "LightGBM" if best_model == lgb_matcher else "LogisticRegression",
    "optimal_threshold": float(best_thresh),
    "validation_macro_f05": float(max(lgb_best_macro_f05, lr_best_macro_f05)),
    "beta": 0.5,
    "feature_names": DEFAULT_FEATURE_NAMES,
}
threshold_path = cfg.CACHE_DIR / "optimal_threshold.json"
with open(threshold_path, "w", encoding="utf-8") as f:
    json.dump(threshold_cfg, f, indent=2)

print(f"Saved best model to:     {best_model_path}")
print(f"Saved threshold info to: {threshold_path}")
print("Turn 5 ML Matching & Validation Complete!")